In [1]:
from huggingface_hub import notebook_login

notebook_login()

### Dataset

In [7]:
from datasets import load_dataset

billsum_raw = load_dataset("FiscalNote/billsum", split="ca_test")
billsum_raw

Dataset({
    features: ['text', 'summary', 'title'],
    num_rows: 1237
})

In [8]:
billsum = billsum_raw.train_test_split(test_size=0.2)
billsum["train"][0]

{'text': 'The people of the State of California do enact as follows:\n\n\nSECTION 1.\nThis act shall be known, and may be cited, as the Identity Theft Resolution Act.\nSEC. 2.\nSection 1785.16.2 of the Civil Code is amended to read:\n1785.16.2.\n(a) No creditor may sell a consumer debt to a debt collector, as defined in 15 U.S.C. Sec. 1692a, if the consumer is a victim of identity theft, as defined in Section 1798.2, and with respect to that debt, the creditor has received notice pursuant to subdivision (k) of Section 1785.16 or paragraph (2) of subdivision (g) of Section 1788.18.\n(b) Subdivision (a) does not apply to a creditor’s sale of a debt to a subsidiary or affiliate of the creditor, if, with respect to that debt, the subsidiary or affiliate does not take any action to collect the debt.\n(c) For the purposes of this section, the requirement in 15 U.S.C. Sec. 1692a, that a person must use an instrumentality of interstate commerce or the mails in the collection of any debt to be 

### Preprocess

In [9]:
from transformers import AutoTokenizer

checkpoint = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

prefix = "summarize: "

def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [10]:
tokenized_billsum = billsum.map(preprocess_function, batched=True)
tokenized_billsum["train"][0]

Map:   0%|          | 0/989 [00:00<?, ? examples/s]

Map:   0%|          | 0/248 [00:00<?, ? examples/s]

{'text': 'The people of the State of California do enact as follows:\n\n\nSECTION 1.\nThis act shall be known, and may be cited, as the Identity Theft Resolution Act.\nSEC. 2.\nSection 1785.16.2 of the Civil Code is amended to read:\n1785.16.2.\n(a) No creditor may sell a consumer debt to a debt collector, as defined in 15 U.S.C. Sec. 1692a, if the consumer is a victim of identity theft, as defined in Section 1798.2, and with respect to that debt, the creditor has received notice pursuant to subdivision (k) of Section 1785.16 or paragraph (2) of subdivision (g) of Section 1788.18.\n(b) Subdivision (a) does not apply to a creditor’s sale of a debt to a subsidiary or affiliate of the creditor, if, with respect to that debt, the subsidiary or affiliate does not take any action to collect the debt.\n(c) For the purposes of this section, the requirement in 15 U.S.C. Sec. 1692a, that a person must use an instrumentality of interstate commerce or the mails in the collection of any debt to be 

### DataCollator

In [11]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=checkpoint)

### Evaluate

In [27]:
import evaluate

rouge = evaluate.load("rouge")

In [28]:
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

### Train

In [29]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_billsum_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=False, #change to bf16=True for XPU
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_billsum["train"],
    eval_dataset=tokenized_billsum["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,2.796954,0.126000,0.036900,0.105000,0.104700,19.064500
2,No log,2.588528,0.132700,0.045600,0.111400,0.111400,19.000000
3,No log,2.524350,0.139100,0.048300,0.114900,0.114900,19.000000
4,No log,2.507337,0.139700,0.048900,0.114700,0.114700,19.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/hedon/mycode/ai/daedalus/workspaces/projects/hugging-face-llm-course/topics/lora-feedback-loop/demo/hugging-face-course-learning/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=248, training_loss=3.022557412424395, metrics={'train_runtime': 701.5058, 'train_samples_per_second': 5.639, 'train_steps_per_second': 0.354, 'total_flos': 1070824333246464.0, 'train_loss': 3.022557412424395, 'epoch': 4.0})

### Inference

In [30]:
text = "summarize: The Inflation Reduction Act lowers prescription drug costs, health care costs, and energy costs. It's the most aggressive action on tackling the climate crisis in American history, which will lift up American workers and create good-paying, union jobs across the country. It'll lower the deficit and ask the ultra-wealthy and corporations to pay their fair share. And no one making under $400,000 per year will pay a penny more in taxes."

In [37]:
from transformers import AutoModelForSeq2SeqLM

model_dir = "my_awesome_billsum_model/checkpoint-248"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
inputs = tokenizer(text, return_tensors="pt").input_ids

model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
outputs = model.generate(inputs, max_new_tokens=100, do_sample=False)
tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"the Inflation Reduction Act lowers prescription drug costs, health care costs, and energy costs. it's the most aggressive action on tackling the climate crisis in American history. it'll ask the ultra-wealthy and corporations to pay their fair share."